In [1]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
import os
load_dotenv()

# Parallel chains

True

In [2]:
# initialize the model

repo_id = "meta-llama/Llama-3.1-8B-Instruct"

llm = HuggingFaceEndpoint(
    repo_id= repo_id,
    max_new_tokens= 50,
    temperature= 0.5,
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

model = ChatHuggingFace(llm = llm)

# We will use same model for everything

e:\AI-LLMs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Using Structured Output Parser

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel

# we need parser, prompts

parser = StrOutputParser()

# Notes prompt

template1 = PromptTemplate(
    template="Give me short notes based on  : {text} "
    , input_variables=["text"]
)

# Quiz prompt

template2 = PromptTemplate(
    template="Give me 5 point quiz based on  : {text} "
    , input_variables=["text"]
)

# Merging prompt

template3 = PromptTemplate(
    template="Merge these two things a notes :{notes} and a quiz :{quiz} "
    , input_variables=["notes","quiz"]
)

notes_chain = template1 | model | parser
quiz_chain = template2 | model | parser


parallel_chain = RunnableParallel(
    {
        "notes": notes_chain,
        "quiz": quiz_chain
    }
)

merge_chain = template3 | model |parser
total_chain = parallel_chain | merge_chain

text = """Types of Runnables
        Runnable can be broadly classified into two categories: Task-Specific Runnables and Runnable Primitives.

        Task-Specific Runnables:
        These are components that do a particular job. They are the main workers in our pipeline. These were originally separate tools, but LangChain has made them runnable so they all follow the same rules and can connect easily in a pipeline. Examples—

        A PromptTemplate that formats user input into a prompt.
        An LLM (like ChatOpenAI) that generates a response.
        A Retriever that fetches relevant documents from a database.
        Runnable Primitives
        These are helper tools that connect, modify, or control how task-specific runnables work together. These don’t do AI tasks themselves, but they help you arrange and control how the AI tasks work together."""

response = total_chain.invoke({"text":text}) # For invoke, 'input' is compulsory, hence just pass dummy {}

print(response)

Here are the merged notes with the quiz questions and answers:

**Types of Runnables**

**1. Task-Specific Runnables**
	* Do a particular job (main workers in the pipeline)
	* Examples: PromptTemplate, LLM, Retriever
	* Primary function: To do a particular job in the pipeline
	* Key characteristic: They are the main workers in the pipeline

**2. Runnable Primitives**
	* Helper tools that connect, modify, or control task-specific runnables
	* Don't do AI tasks themselves, but help arrange and control AI tasks
	* Primary function: To arrange and control how AI tasks work together
	* Examples: Thread (that runs in the background)

**Quiz: Types of Runnables**

**1. What is the primary function of Task-Specific Runnables?**
a) To connect and modify other runnables
b) To do a particular job in the pipeline
c) To control how runnables work together
d) To fetch documents from a database

Answer: b) To do a particular job in the pipeline

**2. Which of the following is an example of a Task-Spe

In [ ]:
# pip install pygraphviz

In [11]:
total_chain.get_graph().print_ascii()

              +---------------------------+              
              | Parallel<notes,quiz>Input |              
              +---------------------------+              
                  ***               ***                  
               ***                     ***               
             **                           **             
+----------------+                    +----------------+ 
| PromptTemplate |                    | PromptTemplate | 
+----------------+                    +----------------+ 
          *                                   *          
          *                                   *          
          *                                   *          
+-----------------+                  +-----------------+ 
| ChatHuggingFace |                  | ChatHuggingFace | 
+-----------------+                  +-----------------+ 
          *                                   *          
          *                                   *          
          *   